## Hub-Spoke Design Pattern with Microsoft Agent Framework

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.10

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Creating the Foundry Chat Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential

client = FoundryChatClient(
    project_endpoint = foundry_project_endpoint,
    model = model_deployment_name,
    credential = credential
)

await client.configure_azure_monitor(
    enable_sensitive_data=True
)

### Define the Web Search Tool

In [ ]:
web_search_tool = FoundryChatClient.get_web_search_tool()

### Create the Flight Spoke

In [ ]:
flight_agent = Agent(
    client=client,

    name="FlightAgent",

    description=(
        "Specialist agent for researching and recommending "
        "current flight options."
    ),

    instructions="""
    You are a flight planning specialist.

    Use web search to research current flight routes,
    airlines, schedules, and other relevant travel information.

    Your responsibility is ONLY flight planning.

    Do not plan hotels or sightseeing activities.

    Return a concise flight recommendation to the
    travel-planning hub.
    """,

    tools=[
        web_search_tool
    ]
)

### Create the Hotel Spoke

In [ ]:
hotel_agent = Agent(
    client=client,

    name="HotelAgent",

    description=(
        "Specialist agent for recommending accommodation "
        "based on destination, budget, and traveler requirements."
    ),

    instructions="""
    You are an accommodation specialist.

    Your responsibility is ONLY accommodation planning.

    When given a travel request:
    - Identify the destination
    - Consider the traveler's budget
    - Recommend suitable areas to stay
    - Suggest appropriate hotel categories
    - Explain why the accommodation fits the traveler

    Do not plan flights or sightseeing activities.

    Return a concise accommodation recommendation
    for the travel planner.
    """,

    tools = [
        web_search_tool
    ]
)

### Create the Activities Spoke

In [ ]:
activities_agent = Agent(
    client=client,

    name="ActivitiesAgent",

    description=(
        "Specialist agent for recommending attractions, "
        "activities, and sightseeing plans."
    ),

    instructions="""
    You are a destination activities specialist.

    Your responsibility is ONLY activities and sightseeing.

    When given a destination:
    - Recommend important attractions
    - Suggest activities
    - Group activities logically
    - Consider the traveler's interests
    - Avoid unrealistic schedules

    Do not recommend flights or hotels.

    Return a concise activity plan for the travel planner.
    """
)

### Convert the Spokes into Callable Tools

In [ ]:
flight_tool = flight_agent.as_tool(
    name="flight_specialist",
    description=(
        "Delegate flight planning and air travel questions "
        "to the flight specialist."
    )
)

hotel_tool = hotel_agent.as_tool(
    name="hotel_specialist",
    description=(
        "Delegate hotel and accommodation planning "
        "to the accommodation specialist."
    )
)

activities_tool = activities_agent.as_tool(
    name="activities_specialist",
    description=(
        "Delegate sightseeing and activity planning "
        "to the destination activities specialist."
    )
)

### Create the Hub Agent

In [ ]:
hub_agent = Agent(
    client=client,

    name="TravelPlanningHub",

    description=(
        "Coordinates specialist travel agents to create "
        "complete travel plans."
    ),

    instructions="""
    You are the hub agent for a multi-agent travel planning system.

    You coordinate specialized spoke agents.

    Available specialists:

    1. flight_specialist
       Use for flights and air travel.

    2. hotel_specialist
       Use for hotels and accommodation.

    3. activities_specialist
       Use for sightseeing and activities.

    Your job is orchestration, not specialist analysis.

    Determine which specialists are required from the user's request.

    Delegate appropriate tasks to the specialists.

    When multiple independent specialists are required,
    you may invoke them as appropriate.

    After receiving their results, synthesize them into one
    coherent travel plan.

    Do not invent specialist results when an appropriate
    specialist can be called.
    """,

    tools=[
        flight_tool,
        hotel_tool,
        activities_tool
    ]
)

### Test your Multi-Agent Setup

In [ ]:
session = hub_agent.create_session()

In [ ]:
query = """
Plan a 5-day trip from Mumbai to Tokyo.

I want:

- flight guidance
- a good area to stay
- sightseeing recommendations

I enjoy technology, Japanese culture and food.

My accommodation budget is moderate.
"""

response = await hub_agent.run(query, session = session)

print(response.text)

In [ ]:
query = """
Actually, I would prefer to spend less on accommodation
and use the savings for better food experiences.

Can you revise the plan accordingly?

Also, suggest which day would be best for visiting
technology-focused attractions.
"""

response = await hub_agent.run(
    query,
    session=session
)

print(response.text)